# Parallel uprid balance DDS load

Manual notebook for loading `sandbox.balance_ostatok_uprid` by date and parallel `personal_account_id` ranges.

## Imports

In [ ]:
import datetime
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timedelta

import clickhouse_connect

## ClickHouse client

In [ ]:
def get_client():
    return clickhouse_connect.get_client(
        host="tp17.wb-bank.ru",
        port=443,
        username="holdzhgonov.a",
        secure=True,
        verify=False,
        client_cert="/home/jovyan/tsh/clickhouse-prod.crt",
        client_cert_key="/home/jovyan/tsh/clickhouse-prod.key",
        connect_timeout=60,
        send_receive_timeout=1800,
    )

In [ ]:
client_test = get_client()
print(client_test.query("select 1").result_rows[0][0])

## Parameters

In [ ]:
CLICKHOUSE_DB = "sandbox"
CLICKHOUSE_TABLE_LOG = "balance_ostatok_uprid"
CLICKHOUSE_TABLE_LOG_LOCAL = f"{CLICKHOUSE_TABLE_LOG}_local"

START_DATE = "2026-05-20"
DAYS = 9
NUM_STEPS = 12
MAX_WORKERS = 6
PA_START_ID = 0

## Helpers

In [ ]:
def get_date_list(start_date: str, days: int) -> list[str]:
    if days <= 0:
        raise ValueError("DAYS must be a positive integer")
    start = datetime.datetime.strptime(start_date, "%Y-%m-%d")
    return [(start + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(days)]


def build_pai_ranges(max_pai: int, num_steps: int, start_id: int = 0) -> list[tuple[int, int]]:
    if num_steps <= 0:
        raise ValueError("NUM_STEPS must be a positive integer")
    if start_id < 0:
        raise ValueError("start_id must be non-negative")

    end_boundary = int(max_pai) + 1
    if end_boundary <= start_id:
        return []

    total_ids = end_boundary - start_id
    step_size = max(1, (total_ids + num_steps - 1) // num_steps)
    return [
        (range_start, min(range_start + step_size, end_boundary))
        for range_start in range(start_id, end_boundary, step_size)
    ]


def get_max_pai(dt: str) -> int:
    client = get_client()
    query = f"""
        SELECT max(personal_account_id)
        FROM abs_clients.abs_wallet_info_uprid AS awi
        WHERE toDate(open_dt) < toDate('{dt}') + interval 1 day
    """
    result = client.query(query).result_rows
    max_pai = result[0][0] if result and result[0] else None
    if max_pai is None:
        raise ValueError(f"No personal_account_id found through {dt}")
    return int(max_pai)

## Insert SQL

In [ ]:
def s_query(dt: str, id_start: int, id_end: int) -> str:
    return f"""
INSERT INTO {CLICKHOUSE_DB}.{CLICKHOUSE_TABLE_LOG}(
        personal_account_id,
        wbuser_id,
        day,
        balance_by_day,
        balance,
        debit_amount,
        credit_amount,
        debit_operations_count,
        credit_operations_count,
        ch_processed_at
    )
    WITH
        {id_start} AS id_start,
        {id_end} AS id_end,
        toDate('{dt}') AS target_day,
    wallet_info AS (
        SELECT
            awi.personal_account_id,
            argMax(awi.wallet_id, awi.ch_processed_at) AS wallet_id,
            argMax(awi.wbuser_id, awi.ch_processed_at) AS wbuser_id
        FROM abs_clients.abs_wallet_info_uprid AS awi
        WHERE awi.open_dt < target_day + interval 1 day
            AND (awi.close_dt >= target_day OR awi.close_dt IS NULL)
            AND awi.personal_account_id >= id_start
            AND awi.personal_account_id < id_end
        GROUP BY awi.personal_account_id
    ),
    last_data AS (
        SELECT
            bo.personal_account_id,
            max(bo.day) AS last_day,
            argMax(bo.balance, bo.ch_processed_at) AS last_balance
        FROM sandbox.balance_ostatok_uprid AS bo
        WHERE bo.day = target_day - interval 1 day
            AND bo.personal_account_id >= id_start
            AND bo.personal_account_id < id_end
        GROUP BY bo.personal_account_id
    ),
    daily_aggregated AS (
        SELECT
            ba.personal_account_id,
            max(ba.credit_amount) AS credit_amount_max,
            max(ba.debit_amount) AS debit_amount_max,
            max(ba.credit_operations_count) AS credit_operations_count_max,
            max(ba.debit_operations_count) AS debit_operations_count_max,
            max(ba.credit_amount) - max(ba.debit_amount) AS daily_balance
        FROM abs_accounting.balance_all AS ba
        WHERE ba.day = target_day
            AND ba.personal_account_id >= id_start
            AND ba.personal_account_id < id_end
        GROUP BY ba.personal_account_id
    )
    SELECT
        wi.personal_account_id AS personal_account_id,
        wi.wbuser_id AS wbuser_id,
        target_day AS day,
        coalesce(da.daily_balance, 0) AS balance_by_day,
        coalesce(ld.last_balance, 0) + coalesce(da.daily_balance, 0) AS balance,
        coalesce(da.debit_amount_max, 0) AS debit_amount,
        coalesce(da.credit_amount_max, 0) AS credit_amount,
        coalesce(da.debit_operations_count_max, 0) AS debit_operations_count,
        coalesce(da.credit_operations_count_max, 0) AS credit_operations_count,
        now() AS ch_processed_at
    FROM wallet_info AS wi
        LEFT ANY JOIN daily_aggregated AS da ON da.personal_account_id = wi.personal_account_id
        LEFT ANY JOIN last_data AS ld ON ld.personal_account_id = wi.personal_account_id
    """

## Parallel execution

In [ ]:
def process_pai_range(dt: str, pai_range: tuple[int, int]) -> tuple[int, int, float]:
    id_start, id_end = pai_range
    client = get_client()
    print(f"START | date={dt}; personal_account_id range=[{id_start}:{id_end})")
    start_batch = time.time()
    client.query(s_query(dt, id_start, id_end))
    elapsed = time.time() - start_batch
    print(f"DONE  | date={dt}; personal_account_id range=[{id_start}:{id_end}) in {elapsed:.2f} sec.")
    return id_start, id_end, elapsed


def process_day(dt: str) -> None:
    start_day = time.time()
    client = get_client()
    print(f"{dt} | START")

    drop_partition_sql = f"""
        ALTER TABLE {CLICKHOUSE_DB}.{CLICKHOUSE_TABLE_LOG_LOCAL}
        ON CLUSTER 'clickhouse'
        DROP PARTITION '{dt}'
    """
    client.query(drop_partition_sql)
    print(f"{dt} | Dropped partition {dt}")

    max_pai = get_max_pai(dt)
    pai_ranges = build_pai_ranges(max_pai, NUM_STEPS, PA_START_ID)
    print(f"{dt} | max_pai={max_pai}; NUM_STEPS={NUM_STEPS}; ranges={pai_ranges}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_pai_range, dt, pai_range) for pai_range in pai_ranges]
        for future in as_completed(futures):
            print(future.result())

    count_select = f"""
        SELECT count()
        FROM {CLICKHOUSE_DB}.{CLICKHOUSE_TABLE_LOG}
        WHERE toDate(day) = toDate('{dt}')
    """
    rows_count = client.query(count_select).result_rows[0][0]
    elapsed_day = time.time() - start_day
    print(f"{dt} | rows={rows_count}; elapsed={elapsed_day:.2f} sec.")

## Main

In [ ]:
date_list_str = get_date_list(START_DATE, DAYS)
date_list_str

In [ ]:
for date_iter in date_list_str:
    process_day(date_iter)

## Manual validation queries

In [ ]:
validation_day = date_list_str[0]

print(f"""
SELECT count()
FROM {CLICKHOUSE_DB}.{CLICKHOUSE_TABLE_LOG}
WHERE day = toDate('{validation_day}');

SELECT personal_account_id, day, count() AS rows_count
FROM {CLICKHOUSE_DB}.{CLICKHOUSE_TABLE_LOG}
WHERE day = toDate('{validation_day}')
GROUP BY personal_account_id, day
HAVING rows_count > 1
LIMIT 100;

SELECT
    countIf(balance IS NULL) AS null_balance,
    countIf(balance_by_day IS NULL) AS null_balance_by_day,
    count() AS rows_total
FROM {CLICKHOUSE_DB}.{CLICKHOUSE_TABLE_LOG}
WHERE day = toDate('{validation_day}');
""")